# Reto 06 — SOLUCION

No liberar hasta cerrar el modulo.

In [ ]:
from pathlib import Path
import pandas as pd
import sqlite3

CARPETA = Path("../../../_RECURSOS/datasets/ventas_mensuales")
assert CARPETA.exists(), f"No encuentro {CARPETA.resolve()}"

## 1 — Listar

`glob` devuelve un generador en orden arbitrario del sistema de archivos. `sorted()` no es
cosmetico: sin el, el orden de consolidacion cambia entre Windows y Linux.

In [ ]:
archivos = sorted(CARPETA.glob("*.csv"))

assert len(archivos) == 12
print("\n".join(a.name for a in archivos))

## 2 — Consolidar

`pd.concat` dentro del bucle es O(n²): copia todo lo acumulado en cada vuelta.
Con 12 archivos no se nota; con 5.000 el proceso no termina. Se acumula en lista y
se concatena una vez.

In [ ]:
partes = []
for archivo in archivos:
    df = pd.read_csv(archivo, sep=";", decimal=",", encoding="utf-8")
    df["archivo_origen"] = archivo.name
    partes.append(df)

todo = pd.concat(partes, ignore_index=True)

assert len(todo) == 491 and todo["archivo_origen"].nunique() == 12
print("OK 2 —", todo.shape)

## 3 — Limpiar, como funcion

`df.copy()` en la primera linea: sin el, la funcion muta el DataFrame que le pasaron y
el alumno pierde media hora buscando por que los datos cambian solos.

In [ ]:
def limpiar(df):
    """Deduplica, normaliza canal, tipa fecha y recalcula importes nulos."""
    df = df.copy()
    df = df.drop_duplicates(
        subset=[c for c in df.columns if c != "archivo_origen"]
    ).reset_index(drop=True)
    df["canal"] = df["canal"].str.strip().str.title()
    df["fecha"] = pd.to_datetime(df["fecha"], format="mixed", dayfirst=True)
    calculado = (df["cantidad"] * df["precio_unitario"]
                 * (1 - df["descuento_pct"] / 100)).round(2)
    df["importe"] = df["importe"].fillna(calculado)
    return df


limpio = limpiar(todo)

assert len(limpio) == 482
assert sorted(limpio["canal"].unique()) == ["Telefono", "Tienda", "Web"]
assert limpio["fecha"].dtype.kind == "M" and limpio["importe"].isna().sum() == 0
print("OK 3 —", limpio.shape)

> **Detalle fino:** el `drop_duplicates` excluye `archivo_origen` del `subset`. Si no,
> dos filas identicas que vinieran de meses distintos no se detectarian como duplicadas.
> Que sea lo correcto depende del negocio — aqui se asume que el `id_venta` es unico.

## 4 — SQLite

`if_exists="append"` sumaria 482 filas mas cada vez que se ejecuta la celda. En un proceso
que corre todos los meses, eso es duplicar la base en silencio.
`"replace"` hace la carga **idempotente**: correrla dos veces da el mismo resultado que una.

In [ ]:
con = sqlite3.connect("tienda.db")
limpio.to_sql("ventas", con, if_exists="replace", index=False)

n = pd.read_sql("SELECT COUNT(*) AS n FROM ventas", con)["n"][0]
assert n == 482
print("OK 4 — tabla ventas con", n, "filas")

## 5 — SQL

In [ ]:
q1 = """
SELECT ciudad,
       ROUND(SUM(importe), 2) AS total,
       COUNT(*)               AS n_ventas
FROM ventas
WHERE cantidad > 0
GROUP BY ciudad
ORDER BY total DESC
LIMIT 5
"""
top_ciudades = pd.read_sql(q1, con)
print(top_ciudades.to_string(index=False))

In [ ]:
q2 = """
SELECT strftime('%Y-%m', fecha) AS mes,
       canal,
       ROUND(SUM(importe), 2)   AS total
FROM ventas
WHERE cantidad > 0
GROUP BY mes, canal
ORDER BY mes, canal
"""
por_canal_mes = pd.read_sql(q2, con)
print(por_canal_mes.head(12).to_string(index=False))

In [ ]:
q3 = """
SELECT id_cliente,
       ROUND(SUM(importe), 2) AS total,
       COUNT(*)               AS n_compras
FROM ventas
WHERE cantidad > 0
GROUP BY id_cliente
ORDER BY total DESC
LIMIT 3
"""
print(pd.read_sql(q3, con).to_string(index=False))

## 6 — Reporte a disco

In [ ]:
salida = Path("reporte_ciudades.txt")
lineas = ["TOP 5 CIUDADES POR FACTURACION", "=" * 40]
for _, r in top_ciudades.iterrows():
    lineas.append(f"{r['ciudad']:<12} S/ {r['total']:>12,.2f}   ({int(r['n_ventas'])} ventas)")

salida.write_text("\n".join(lineas), encoding="utf-8")

assert salida.exists()
print(salida.read_text(encoding="utf-8"))
con.close()

## 7 — Idempotencia

Correr el proceso dos veces debe dejar la base **exactamente igual**. Es la propiedad que
separa un script de un pipeline.

In [ ]:
def procesar_carpeta(carpeta, db="tienda.db"):
    """Consolida, limpia y carga la carpeta a SQLite. Idempotente."""
    archivos = sorted(Path(carpeta).glob("*.csv"))
    if not archivos:
        raise FileNotFoundError(f"no hay CSV en {carpeta}")
    partes = []
    for a in archivos:
        df = pd.read_csv(a, sep=";", decimal=",", encoding="utf-8")
        df["archivo_origen"] = a.name
        partes.append(df)
    limpio = limpiar(pd.concat(partes, ignore_index=True))
    with sqlite3.connect(db) as con:
        limpio.to_sql("ventas", con, if_exists="replace", index=False)
    return len(limpio)


print(procesar_carpeta(CARPETA))
print(procesar_carpeta(CARPETA))

with sqlite3.connect("tienda.db") as con:
    n = pd.read_sql("SELECT COUNT(*) AS n FROM ventas", con)["n"][0]
assert n == 482, f"no es idempotente: {n} filas"
print("OK 7 — idempotente, sigue en", n)

---
### Errores tipicos al corregir

- **Nombres de archivo hardcodeados** o un `range(1, 13)` construyendo los nombres. Rompe en cuanto llega un mes nuevo o falta uno.
- **`os.listdir()` sin filtrar**: se traga el `desktop.ini` y truena en `read_csv`.
- **`pd.concat` dentro del bucle.**
- **Rutas armadas con `"carpeta" + "\\" + nombre`.** `Path` existe justo para no hacer eso.
- **`if_exists="append"`** y la base creciendo en cada ejecucion.
- **Resolver el punto 5 con `groupby` de pandas** y envolverlo en un `read_sql` decorativo.
- **No cerrar la conexion** (`con.close()` o `with`). En Windows deja `tienda.db` bloqueado.